In [58]:
import pandas as pd
import numpy as np

df = pd.read_csv("hw_submissions.csv")
df

,student,section,assignment,due,submitted,score,max_points
0,Ama,1,HW1,2026-09-04 23:59,2026-09-04 21:15,18,20
1,Ama,1,HW2,2026-09-11 23:59,2026-09-12 01:30,15,20
2,Ama,1,HW3,2026-09-18 23:59,2026-09-18 23:40,19,20
3,Ama,1,HW4,2026-09-25 23:59,2026-09-26 08:05,12,20
4,Ben,2,HW1,2026-09-04 23:59,2026-09-03 14:20,20,20
5,Ben,2,HW2,2026-09-11 23:59,2026-09-11 16:45,17,20
6,Cruz,3,HW1,2026-09-04 23:59,2026-09-05 00:20,11,20
7,Cruz,3,HW2,2026-09-11 23:59,2026-09-11 22:10,14,20


In [59]:
fe = pd.DataFrame()

In [60]:
fe["student"] = df["student"]
fe["pct"] = (df["score"] / 20) * 100

fe["pct"] = fe["pct"].astype(str).astype(float)

In [61]:

df["submitted"] = pd.to_datetime(df["submitted"])
df["due"] = pd.to_datetime(df["due"])

fe["hours_early"] = (df["due"] - df["submitted"]).dt.total_seconds() / 3600
fe["is_late"] = fe["hours_early"] < 0
fe["weekday"] = df["submitted"].dt.day_name()
fe["hour"] = df["submitted"].dt.hour
fe["is_night"] = (df["submitted"].dt.hour >= 22) | (df["submitted"].dt.hour < 6)

In [63]:
fe["n_submissions"] = df.groupby("student")["assignment"].transform("count")
fe["student_mean_pct"] = fe.groupby("student")["pct"].transform("mean")

fe

,student,pct,hours_early,is_late,weekday,hour,is_night,n_submissions,student_mean_pct
0,Ama,90.0,2.733333,False,Friday,21,False,4,80.0
1,Ama,75.0,-1.516667,True,Saturday,1,True,4,80.0
2,Ama,95.0,0.316667,False,Friday,23,True,4,80.0
3,Ama,60.0,-8.100000,True,Saturday,8,False,4,80.0
4,Ben,100.0,33.650000,False,Thursday,14,False,2,92.5
5,Ben,85.0,7.233333,False,Friday,16,False,2,92.5
6,Cruz,55.0,-0.350000,True,Saturday,0,True,2,62.5
7,Cruz,70.0,1.816667,False,Friday,22,True,2,62.5


In [64]:
bins = [0, 60, 70, 80, 90, 101]
labels = ["F", "D", "C", "B", "A"]

fe["pct"] = pd.cut(fe["pct"], bins=bins, labels=labels, right=False)
fe

,student,pct,hours_early,is_late,weekday,hour,is_night,n_submissions,student_mean_pct
0,Ama,A,2.733333,False,Friday,21,False,4,80.0
1,Ama,C,-1.516667,True,Saturday,1,True,4,80.0
2,Ama,A,0.316667,False,Friday,23,True,4,80.0
3,Ama,D,-8.100000,True,Saturday,8,False,4,80.0
4,Ben,A,33.650000,False,Thursday,14,False,2,92.5
5,Ben,B,7.233333,False,Friday,16,False,2,92.5
6,Cruz,F,-0.350000,True,Saturday,0,True,2,62.5
7,Cruz,C,1.816667,False,Friday,22,True,2,62.5


In [65]:
 pd.get_dummies(df["section"], prefix="section", drop_first=True)

,section_2,section_3
0,False,False
1,False,False
2,False,False
3,False,False
4,True,False
5,True,False
6,False,True
7,False,True


In [66]:
fe["total_late"] = fe.groupby("student")["is_late"].transform("sum")

running_late = fe.groupby("student")["is_late"].cumsum()

fe["total_late_before"] = running_late - fe["is_late"].astype(int)

In [67]:
fe

,student,pct,hours_early,is_late,weekday,hour,is_night,n_submissions,student_mean_pct,total_late,total_late_before
0,Ama,A,2.733333,False,Friday,21,False,4,80.0,2,0
1,Ama,C,-1.516667,True,Saturday,1,True,4,80.0,2,0
2,Ama,A,0.316667,False,Friday,23,True,4,80.0,2,1
3,Ama,D,-8.100000,True,Saturday,8,False,4,80.0,2,1
4,Ben,A,33.650000,False,Thursday,14,False,2,92.5,0,0
5,Ben,B,7.233333,False,Friday,16,False,2,92.5,0,0
6,Cruz,F,-0.350000,True,Saturday,0,True,2,62.5,1,0
7,Cruz,C,1.816667,False,Friday,22,True,2,62.5,1,1
